# 01 · HTTP, Web APIs & the `requests` Library

**Module 14 – Working with Data & the Web** · Estimated time: 50 min

> Expands 📘 *Requests Module – Working with APIs* (one GET example) into a full lesson. The 📗 bootcamp uses `requests` only for scraping.

### 🎯 What you will learn
- How HTTP works: URLs, methods, status codes, headers, JSON bodies
- `requests`: GET with query parameters, POST/PUT/DELETE, headers, timeouts
- Handling errors properly (`raise_for_status`, exceptions)
- Sessions, authentication tokens (kept in environment variables), pagination, rate limits
- The standard-library alternative `urllib`; async alternatives (`httpx`)

> 🌐 This notebook makes real requests to public test APIs (`httpbin.org`, `api.github.com`). If a service is down when you run it, try again later.

## 1. HTTP in two minutes

Your program (the **client**) sends a **request**; the **server** returns a **response**.

```
GET https://api.github.com/users/octocat?per_page=5
     └── scheme ┘└──── host ───┘└── path ────┘└─ query string ┘

Request:  method + URL + headers (+ body)
Response: status code + headers + body (often JSON)
```

| Method | Meaning |
|---|---|
| `GET` | read data |
| `POST` | create / submit data |
| `PUT` / `PATCH` | replace / partially update |
| `DELETE` | delete |

| Status | Meaning |
|---|---|
| `2xx` | success (200 OK, 201 Created, 204 No Content) |
| `3xx` | redirect |
| `4xx` | **client** error (400 Bad Request, 401 Unauthorized, 403 Forbidden, 404 Not Found, 429 Too Many Requests) |
| `5xx` | **server** error (500, 502, 503) |

## 2. A first GET request (📘)

`pip install requests` (installed with this course).

In [1]:
import requests

url = "https://api.github.com/users/octocat"
response = requests.get(url, timeout=10)

print(response.status_code)                     # 200
print(response.headers["Content-Type"])
if response.status_code == 200:
    data = response.json()                      # parse the JSON body into a dict
    print(data["name"], "|", data["public_repos"], "public repos")
else:
    print(f"Error: {response.status_code}")

200
application/json; charset=utf-8
The Octocat | 8 public repos


> ✅ **Always pass `timeout=`.** Without it, `requests` can wait *forever* if a server hangs.

## 3. Query parameters, headers

In [2]:
params = {"q": "language:python stars:>50000", "sort": "stars", "per_page": 5}
headers = {"Accept": "application/vnd.github+json", "User-Agent": "python-course-demo"}
r = requests.get("https://api.github.com/search/repositories", params=params, headers=headers, timeout=10)
print(r.url)                                  # requests builds & encodes the query string for you
for repo in r.json()["items"]:
    print(f"{repo['stargazers_count']:>8,} ⭐  {repo['full_name']}")

https://api.github.com/search/repositories?q=language%3Apython+stars%3A%3E50000&sort=stars&per_page=5
 490,747 ⭐  public-apis/public-apis
 399,657 ⭐  EbookFoundation/free-programming-books
 374,868 ⭐  donnemartin/system-design-primer
 328,910 ⭐  vinta/awesome-python
 287,372 ⭐  practical-tutorials/project-based-learning


## 4. Sending data: POST, PUT, DELETE

`httpbin.org` echoes back what you send, which is perfect for learning.

In [3]:
payload = {"name": "Asha", "course": "Python"}
r = requests.post("https://httpbin.org/post", json=payload, timeout=15)   # json= sends a JSON body
print(r.status_code)
print(r.json()["json"])                       # what the server received

r = requests.post("https://httpbin.org/post", data={"user": "asha", "pw": "x"}, timeout=15)   # form data
print(r.json()["form"])

print(requests.put("https://httpbin.org/put", json={"id": 1}, timeout=15).status_code)
print(requests.delete("https://httpbin.org/delete", timeout=15).status_code)

200
{'course': 'Python', 'name': 'Asha'}


{'pw': 'x', 'user': 'asha'}


200


200


## 5. Handling errors properly

In [4]:
def get_json(url, **kwargs):
    try:
        r = requests.get(url, timeout=10, **kwargs)
        r.raise_for_status()                  # raises HTTPError for 4xx/5xx
        return r.json()
    except requests.exceptions.Timeout:
        print("⏱ The server took too long")
    except requests.exceptions.ConnectionError:
        print("🔌 Could not connect (offline? wrong host?)")
    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP error: {e.response.status_code} for {e.request.url}")
    except requests.exceptions.JSONDecodeError:
        print("Response was not JSON")
    return None

print(get_json("https://httpbin.org/status/404"))
print(get_json("https://does-not-exist.invalid"))
print(get_json("https://httpbin.org/json")["slideshow"]["title"])

❌ HTTP error: 404 for https://httpbin.org/status/404
None
🔌 Could not connect (offline? wrong host?)
None


Sample Slide Show


## 6. Authentication & secrets

APIs usually require a key or token, sent in a header. ✅ **Never hard-code secrets** in your code or commit them to Git. Read them from environment variables (or a `.env` file that's in `.gitignore`):

```python
import os
token = os.environ["GITHUB_TOKEN"]                       # set in your shell: export GITHUB_TOKEN=...
headers = {"Authorization": f"Bearer {token}"}
r = requests.get("https://api.github.com/user", headers=headers, timeout=10)
```

In [5]:
r = requests.get("https://httpbin.org/bearer", headers={"Authorization": "Bearer demo-token"}, timeout=15)
print(r.status_code, r.json())
r = requests.get("https://httpbin.org/basic-auth/user/pass", auth=("user", "pass"), timeout=15)   # HTTP basic auth
print(r.status_code, r.json())

200 {'authenticated': True, 'token': 'demo-token'}


200 {'authenticated': True, 'user': 'user'}


## 7. Sessions: reuse connections and shared settings

In [6]:
with requests.Session() as s:
    s.headers.update({"User-Agent": "python-course-demo"})
    for user in ["octocat", "gvanrossum"]:
        info = s.get(f"https://api.github.com/users/{user}", timeout=10).json()
        print(f"{info.get('login')}: {info.get('followers')} followers")

octocat: 24676 followers


gvanrossum: 27193 followers


## 8. Pagination and rate limits

Big result sets come in **pages**. Loop until there are no more. APIs also limit how many calls you can make; check the headers and back off when you get `429`.

In [7]:
r = requests.get("https://api.github.com/users/octocat/repos", params={"per_page": 3, "page": 1}, timeout=10)
print("Rate limit remaining:", r.headers.get("X-RateLimit-Remaining"))
print("Next page link:", r.links.get("next", {}).get("url"))

def fetch_all(url, max_pages=3):
    results, page = [], 1
    while url and page <= max_pages:
        r = requests.get(url, params={"per_page": 5} if page == 1 else None, timeout=10)
        r.raise_for_status()
        results.extend(r.json())
        url = r.links.get("next", {}).get("url")     # GitHub puts the next page in the Link header
        page += 1
    return results

repos = fetch_all("https://api.github.com/users/octocat/repos")
print(len(repos), [r["name"] for r in repos][:6])

Rate limit remaining: 53
Next page link: https://api.github.com/user/583231/repos?per_page=3&page=2


8 ['boysenberry-repo-1', 'git-consortium', 'hello-worId', 'Hello-World', 'linguist', 'octocat.github.io']


## 9. Downloading files

In [8]:
r = requests.get("https://www.python.org/static/img/python-logo.png", timeout=15, stream=True)
r.raise_for_status()
with open("python-logo.png", "wb") as f:
    for chunk in r.iter_content(chunk_size=8192):      # stream big files in chunks
        f.write(chunk)
import os
print(os.path.getsize("python-logo.png"), "bytes saved")

15770 bytes saved


## 10. Without third-party libraries: `urllib`

In [9]:
import json, urllib.request, urllib.parse

query = urllib.parse.urlencode({"name": "Ravi", "lang": "python"})
with urllib.request.urlopen(f"https://httpbin.org/get?{query}", timeout=15) as resp:
    body = json.load(resp)
print(body["args"])

{'lang': 'python', 'name': 'Ravi'}


> 💡 For **async** code (Module 13) use `httpx.AsyncClient` or `aiohttp`. `httpx` also has a synchronous API almost identical to `requests`, plus HTTP/2 support.

## 11. A small project: a reusable API client class

In [10]:
from dataclasses import dataclass

@dataclass
class GitHubUser:
    login: str
    name: str | None
    public_repos: int
    followers: int

class GitHubClient:
    BASE = "https://api.github.com"

    def __init__(self, token: str | None = None):
        self.session = requests.Session()
        self.session.headers["Accept"] = "application/vnd.github+json"
        if token:
            self.session.headers["Authorization"] = f"Bearer {token}"

    def user(self, login: str) -> GitHubUser:
        r = self.session.get(f"{self.BASE}/users/{login}", timeout=10)
        r.raise_for_status()
        d = r.json()
        return GitHubUser(d["login"], d.get("name"), d["public_repos"], d["followers"])

client = GitHubClient(token=os.environ.get("GITHUB_TOKEN"))
print(client.user("octocat"))

GitHubUser(login='octocat', name='The Octocat', public_repos=8, followers=24676)


## 📝 Summary
- HTTP = method + URL + headers (+ body) → status + headers + body.
- `requests.get/post/put/delete(url, params=, json=, headers=, timeout=)`.
- `r.status_code`, `r.json()`, `r.raise_for_status()`; catch `requests.exceptions.*`.
- Keep tokens in environment variables; use `Session` for repeated calls; respect pagination and rate limits.

## 🏋️ Practice
1. Use `https://api.github.com/repos/python/cpython` to print the star count, fork count and open issues.
2. POST a JSON body `{"title": "hello", "done": false}` to `https://httpbin.org/post` and print what the server received.
3. Write `safe_get(url)` that retries up to 3 times on `ConnectionError`/`Timeout` with a 1-second pause.

<details><summary>Show solutions</summary>

```python
import requests, time

d = requests.get("https://api.github.com/repos/python/cpython", timeout=10).json()
print(d["stargazers_count"], d["forks_count"], d["open_issues_count"])

r = requests.post("https://httpbin.org/post", json={"title": "hello", "done": False}, timeout=10)
print(r.json()["json"])

def safe_get(url, retries=3):
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, timeout=5)
            r.raise_for_status()
            return r
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
            if attempt == retries:
                raise
            time.sleep(1)
```
</details>